# Light Curve calculation using PINT

## Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd


plt.rcParams['figure.dpi'] = 400
plt.rcParams['savefig.dpi'] = 400

In [ ]:
dirDir = "/home/users/l/lebrun/"
dataPath = dirDir + "data/"
resultPath = dirDir + "results/"
figPath = dirDir + "figures/"

filename = dataPath + f"DmlNtup_Data_9years_FullPhotonData_AllSky.csv"

## Load dataframe

In [ ]:
df = pd.read_csv(filename, index_col=0)
print(df)

df = df.sort_values(['tt_second', 'tt_milisecond'])

## Center on target

In [ ]:
target = "Vela" # Change this for output filenames
psr_name = "B0833-45"

dR = 3 # degrees, radius of region of interest
lon_target = 263.55
lat_target = -2.79 

lon_min = lon_target - dR
lon_max = lon_target + dR
lat_min = lat_target - dR
lat_max = lat_target + dR

df = df.loc[np.sqrt((df["longitude_deg"] - lon_target)**2 + (df["latitude_deg"]-lat_target)**2) < dR]

## Map target for illustration

In [ ]:
plt.figure(figsize=(8, 6))
plt.hist2d(
    df["longitude_deg"],
    df["latitude_deg"],
    bins=50,  # Adjust number of bins as needed
    cmap='viridis'  # Or any other colormap
)
plt.colorbar(label='Entry count')

# Plot the target coordinate as a red dot
plt.plot(lon_target, lat_target, 'ro', markersize=8, label='Target')

plt.xlabel('Longitude (deg)')
plt.ylabel('Latitude (deg)')
plt.title(f'2D Histogram of Entry Counts in {target} ROI')
plt.show()

## Computation of MJD times in TT

In [ ]:
from astropy.time import Time, TimeDelta
import astropy.units as u
from astropy.coordinates import EarthLocation

LEAP_SECONDS = [
    "2015-07-01T00:00:00",  # +1 leap second
    "2017-01-01T00:00:00",  # +1 leap second
]

def add_leap_seconds(mjd1, mjd2):
    times = Time(mjd1, mjd2, scale='tt', format='mjd')

    total_correction = np.zeros(len(times))  # seconds

    for leap in LEAP_SECONDS:
        time_leap = Time(leap, scale='tt')
        total_correction += (times > time_leap).astype(int) # Adds 1 or 0 -> Plays on boolean type

    corrected_times = times + TimeDelta(total_correction, format='sec')

    mjd1_new = corrected_times.tt.jd1 - 2400000.5
    mjd2_new = corrected_times.tt.jd2

    return mjd1_new, mjd2_new

def met_to_mjd_tt_split(seconds, milliseconds):
    # Total delta since epoch: seconds + milliseconds, kept separate
    seconds = np.asarray(seconds)
    milliseconds = np.asarray(milliseconds)

    # Construct TimeDelta in two parts to preserve precision
    delta_sec = TimeDelta(seconds, format='sec')
    delta_ms  = TimeDelta(milliseconds * 1e-3, format='sec')  # convert ms to s

    # MET epoch in TT 
    epoch = Time("2013-01-01T00:00:00", scale="utc")

    full_time = epoch + delta_sec + delta_ms

    # Get split MJD
    mjd1 = full_time.tt.jd1 - 2400000.5
    mjd2 = full_time.tt.jd2

    return mjd1, mjd2

def ecef_to_eci(mjd1, mjd2, x, y, z):
    t = Time(mjd1, mjd2, scale='tt', format='mjd')

    if isinstance(x, pd.Series):
        x = x.to_numpy() * u.m
    if isinstance(y, pd.Series):
        y = y.to_numpy() * u.m
    if isinstance(z, pd.Series):
        z = z.to_numpy() * u.m

    loc_ecef = EarthLocation(x=x, y=y, z=z)

    pos_gcrs, vel_gcrs = loc_ecef.get_gcrs_posvel(t)


    # TODO when we have sat velocities, figure out whether we can correct them too
    return pos_gcrs.get_xyz()




## Create MJD TT timestamps and save orbital file

In [ ]:
df["MJD1"], df["MJD2"] = met_to_mjd_tt_split(df["tt_second"], df["tt_milisecond"])
df["MJD1"], df["MJD2"] = add_leap_seconds(df["MJD1"].values, df["MJD2"].values)

# df['X_ECI'], df['Y_ECI'], df['Z_ECI'] = ecef_to_eci(df["MJD1"], df["MJD2"],
#                                                      df['tt_sat_position_x'],
#                                                      df['tt_sat_position_y'],
#                                                      df['tt_sat_position_z'])

print(f"Minimal day : {df['MJD1'].min()}, max day : {df['MJD1'].min()}")
# df['MJD2'] = df['tt_milisecond']*1e-3

# Save positions and timestamps 
#df.loc[:, ['MJD1', 'MJD2', 'tt_sat_position_x', 'tt_sat_position_y', 'tt_sat_position_z']].to_csv(
 #   resultPath + f"df_{target}_tt.csv")
df.loc[:, ['MJD1', 'MJD2', 'tt_sat_position_x', 'tt_sat_position_y', 'tt_sat_position_z']].to_csv(
   resultPath + f"df_{target}_tt.csv")

## Use PINT to barycentre the TOAs

In [ ]:
import sys
import os
import pint
import astropy
import scipy

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Current working directory:", os.getcwd())
print("PYTHONPATH:", os.environ.get('PYTHONPATH', 'Not set'))


In [ ]:
from pint.toa import TOAs, get_TOAs_array
from astropy import units as u
from pint.observatory.satellite_obs import SatelliteObs

DAMPE = SatelliteObs("DAMPE", resultPath+f"df_{target}_tt.csv", overwrite=True)

def generate_TOAs(df_: pd.DataFrame, model, bipm=False) -> TOAs :
    
    times = Time(df_["MJD1"].to_numpy(), df_["MJD2"].to_numpy(), format="mjd", scale="tt")
    freqs = []
    error = 0.001 * u.s
    for i, row in df_.iterrows():
            # Time of observation
            freq = 0 # Means inf (cf. Paul)
            freqs.append(freq)

    return get_TOAs_array(times, "DAMPE", errors=error, freqs=freqs, ephem="de440s",
                          include_bipm=False, planets=True, model=model)


## Cut on validity of Ephemerids

In [ ]:
from pint.models.model_builder import get_model
import os

# Load par file in directory of target
parDirName = dirDir + f"data/{target}/"

# HERE you can enter your custom path to a .par file. Otherwise the script will look
# into data/{target} for a .par file
filename = None 

if filename is None :

    try:
        for file in os.listdir(parDirName[:-1]):
            if file.endswith(".par"):
                filename = os.path.join(parDirName, file)
    except FileNotFoundError as e :
        print(f"Error : no par found file in subdirectory {parDirName}. Target : {target}")
        raise e
#filename = resultPath + "Vela.par"
# Load the timing model
model = get_model(filename)  # path to your .par file

max_ephem_validity = model.FINISH.value 
max_t = Time(max_ephem_validity, format='mjd', scale='tt')
print(f"Max model validity : {max_ephem_validity} MJD i.e. {max_t.to_datetime()}")

df = df.loc[df['MJD1'] < max_ephem_validity]

toas = generate_TOAs(df, model)

phase = model.phase(toas,abs_phase=True)  # Returns a PINT Phase object

# Access fractional phases (between 0 and 1)
frac_phase = phase.frac.value


## Compute phaseogram

In [ ]:
from pint.plot_utils import phaseogram_binned, phaseogram

In [ ]:
times = Time(df["MJD1"].to_numpy(), df["MJD2"].to_numpy(), format="mjd", scale="tt")
figPath = dirDir + "figures/"

ph = frac_phase + 1.0*(frac_phase<0.0)
phaseogram(times, ph, maxphs=2) # Add 0.5 bc phaseogram expects [0, 1)

## CONCLUSIONS
- Timestamps start 01/01/2013 and are given in UTC, without leap seconds
- coordinates are given in GCRS frame
- ephemerids are crucial. TODO : compare with worse ones
- TODO : slight break at 2024 : why ? 


## Plot the light curve

In [ ]:
def centralize_phases(phases):
    print(f"Mean : {np.mean(phases)}")
    phases -= np.mean(phases) # substracts mean and centralize
    return phases % 1
    

frac_phase_ctr = centralize_phases(frac_phase)
df['Phase'] = frac_phase_ctr

In [ ]:
from pyparsing import line
from scipy.signal import find_peaks

fig, ax = plt.subplots()
# Histogram
bins = 96
width = 1/bins
counts, bin_edges = np.histogram(frac_phase_ctr, bins=bins, range=(0, 1))
error = np.sqrt(counts)  # Poisson error on counts
bin_centers = 0.5 * (bin_edges[1:] + bin_edges[:-1])
# Peak detection
peaks, properties = find_peaks(
    counts,
    height=np.max(counts) * 0.2,     # min height (20% of max)
    distance=bins // 10,             # at least 10% of phase away
    prominence=np.max(counts) * 0.05 # to avoid tiny bumps
)

# Prepare label with phase positions
peak_phases = bin_centers[peaks]
peak_phases = np.sort(peak_phases)

line_color='C0'
ax.plot(bin_centers, counts, drawstyle='steps-mid', label='Profile', color=line_color, linewidth=1)
ax.fill_between(bin_centers, counts, step='mid', color=line_color, alpha=0.3)  # fill under curve
ax.plot(bin_centers[peaks], counts[peaks], "rx", label='Detected Peaks')
ax.errorbar(bin_centers, counts, yerr=error, color=None, fmt="none",
            ecolor='C0', elinewidth=1, capsize=2, zorder = -1)
ax.set_xlabel("Pulse Phase")
ax.set_ylabel("Counts")
# Label each peak next to its red cross
for i, (x, y) in enumerate(zip(peak_phases, counts[peaks]), start=1):
    ax.text(
        x, y + np.max(counts) * 0.045,  # small horizontal offset
        rf"$\phi_{i} = {x:.3f}$",
        color='red',
        ha='center',
        va='bottom',
        fontsize=9,
        rotation=0,
        zorder = 1
    )
ax.set_title(f"{target} Folded Profile")
fig.savefig(figPath + f"lightCurve_{target}.png")
plt.show()

# TODO compute Poisson errorbars + variable width bins ? With 50 counts ? 

## TODO : variable width bins
Reproduce Abdo et aL 2007 : Fermi vela

In [ ]:
# Sort phases
sorted_phases = np.sort(frac_phase_ctr)
counts_per_bin = 50
n_bins = len(sorted_phases) // counts_per_bin

bin_edges = [sorted_phases[0]]
for i in range(1, n_bins):
    bin_edges.append(sorted_phases[i * counts_per_bin])
bin_edges.append(sorted_phases[-1])

# Compute flux and error per bin
fluxes = []
errors = []
bin_centers = []
bin_widths = []

for i in range(len(bin_edges) - 1):
    start = bin_edges[i]
    end = bin_edges[i + 1]
    width = end - start
    flux = counts_per_bin / width
    error = np.sqrt(counts_per_bin) / width  # Poisson error on flux

    bin_centers.append((start + end) / 2)
    bin_widths.append(width)
    fluxes.append(flux)
    errors.append(error)

# Normalize to 100 at the max
fluxes = np.array(fluxes)
errors = np.array(errors)
fluxes_normalized = 100 * fluxes / np.max(fluxes)
errors_normalized = 100 * errors / np.max(fluxes)


# Plot
fig, ax = plt.subplots()
ax.bar(bin_centers, fluxes_normalized, width=bin_widths, align='center',
       color='C0', alpha=0.3, edgecolor=None, label='Variable Bin Histogram')
ax.errorbar(bin_centers, fluxes_normalized, yerr=errors_normalized, color='C0',
            ecolor='gray', elinewidth=1, capsize=2)

ax.set_xlabel("Pulse Phase")
ax.set_ylabel("Relative Count Rate")
ax.set_title(f"{target} Folded Profile (Normalized, 50 photons/bin)")
ax.set_xlim(0, 1)
fig.tight_layout()
fig.savefig(figPath + f"lightCurve_variableBins_norm_{target}.png")
plt.show()


## Compute H-Test

In [ ]:

def h_test(phases, max_harmonics=20):
    """Compute the H-test statistic from phases in [0, 1)."""
    n = len(phases)
    z_m = np.zeros(max_harmonics)
    for m in range(1, max_harmonics + 1):
        angle = 2 * np.pi * m * phases
        C_m = np.sum(np.cos(angle))
        S_m = np.sum(np.sin(angle))
        z_m[m - 1] = (C_m**2 + S_m**2) / n

    Z = np.cumsum(z_m)
    H_vals = Z - 4 * np.arange(1, max_harmonics + 1) + 4
    H = np.max(H_vals)
    M = np.argmax(H_vals) + 1
    return H, M

# Usage:
H, M = h_test(frac_phase_ctr)
print(f"H-statistic = {H:.2f}, optimal harmonics = {M}")

In [ ]:

window_size = 20  # days
step_size = 4    # days
min_mjd = df["MJD1"].min()
max_mjd = df["MJD1"].max()

times = []
hs = []

# 3. Slide the window
current = min_mjd
while current + window_size <= max_mjd:
    in_window = df[(df["MJD1"] >= current) & (df["MJD1"] < current + window_size)]
    phases = in_window["Phase"].values
    H, _ = h_test(phases)
    times.append(current + window_size)
    hs.append(H)
    window_size += step_size

# 4. Plot H over time
plt.figure(figsize=(10, 5))
plt.plot(times, hs, marker='o')
plt.xlabel("MJD")
plt.ylabel("H-statistic")
plt.title(f"H-test evolution over time")
plt.grid(True)
plt.tight_layout()
plt.show()

Conclusion : problem in 2024